In [ ]:
import os
import glob
from dotenv import load_dotenv
import gradio as gr
from openai import OpenAI

MODEL = "gpt-4o-mini"

load_dotenv()

openai = OpenAI()

In [ ]:
context = {}

employees = glob.glob("knowledge-base/employees/*")

for employee in employees:
    name = employee.split(' ')[-1][:-3]
    doc = ""
    with open(employee, "r") as f:
        doc = f.read()
    context[name] = doc

context.keys()

In [ ]:
products = glob.glob("knowledge-base/products/*")

for product in products:
    name = product.split("/")[-1][:-3]
    doc = ""
    with open(product, "r") as f:
        doc = f.read()
    context[name] = doc

In [ ]:
context.keys()

In [ ]:
system_message = "You are an expert in answering accurate questions about Insurellm, the Insurance Tech company. Give brief, accurate answers. If you don't know the answer, say so. Do not make anything up if you haven't been provided with relevant context."

In [ ]:
def get_relevant_context(message):
    relevant_context = []
    for context_title, context_details in context.items():
        if context_title in message:
            relevant_context.append(context_details)

    return relevant_context

In [ ]:
get_relevant_context("Who is Avery Lancaster?")

In [ ]:
def add_context(message):
    relevant_context = get_relevant_context(message)
    if relevant_context:
        message += "\n\nThe following additional context might be relevant in answering this question:\n\n"
        for relevant in relevant_context:
            message += relevant + "\n\n"
    return message

In [ ]:
print(add_context("Who is Avery Lancaster?"))

In [ ]:
def chat(message, history):
    messages = [
        {"role": "system", "content": system_message}
    ]
    for user_message, assistant_message in history:
        messages.append({"role": "user", "content": user_message})
        messages.append({"role": "assistant", "content": assistant_message})

    message = add_context(message)
    messages.append({"role": "user", "content" : message})

    stream = openai.chat.completions.create(model=MODEL, messages=messages, stream=True)
    
    response = ""
    for chunk in stream:
        response += chunk.choices[0].delta.content or ""
        yield response

In [ ]:
view = gr.ChatInterface(chat).launch()